# ATLAS Wind conversion

This notebook converts the final integrated wind components into wind speed and wind direction.

Expected inputs are the NetCDF files produced by the wind scaling workflow, for example:

```text
u10_integrated_chile_m11_continental.nc
v10_integrated_chile_m11_continental.nc
```

The output is a single NetCDF file containing:

```text
u10_integrated
v10_integrated
wds_integrated
dir_integrated
```

The output file is then used by the wind plotting notebook.

## 0. Load libraries

Run this cell first. It imports the Python packages used to read NetCDF files, compute wind speed and direction, and save the final product.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import rioxarray  # noqa: F401. Needed to activate the .rio accessor in xarray
import xarray as xr

warnings.filterwarnings("ignore")

## 1. User settings

Edit only this cell for a standard run.

Use `AREA_NAME = "continental"` for the continental area and `AREA_NAME = "islands"` for the island area, if available.

The notebook can process one month or several months. For a single month use, for example, `MONTHS = [11]`.

In [2]:
# Country or area name used in the file names.
COUNTRY = "chile"

# Months to process. Use integers from 1 to 12.
MONTHS = list(range(1, 2))

# Area name used in the file names.
# Typical values are "continental" or "islands".
AREA_NAME = "continental"

# Name of the component variables expected in the input files.
# These names are consistent with the wind scaling notebook.
U_COMPONENT_VAR = "u10_integrated"
V_COMPONENT_VAR = "v10_integrated"

# Names of the variables that will be created by this notebook.
WIND_SPEED_VAR = "wds_integrated"
WIND_DIRECTION_VAR = "dir_integrated"

## 2. Folder configuration

The paths below are relative and anonymous, so the notebook can be shared without exposing local server folders.

Before running the notebook, place the input component files in the expected folder or modify the paths to match your project structure.

In [3]:
# Base project folder.
# By default, this points to a local "data" folder next to the notebook.
BASE_DIR = Path("../data")

# Main ATLAS folder for the selected country.
ATLAS_DIR = BASE_DIR / "atlas_data" / COUNTRY 

# Folder containing the integrated u and v component files produced by the scaling notebook.
INPUT_DIR = ATLAS_DIR / "sub_areas"

# Folder where the converted wind speed and direction file will be written.
OUTPUT_DIR = ATLAS_DIR 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

Input folder: ../data/atlas_data/chile/sub_areas
Output folder: ../data/atlas_data/chile


## 3. Helper functions

These functions make the workflow easier to read.

They check that files exist, open each component dataset, compute wind speed and meteorological wind direction, and save the output NetCDF with light compression.

In [4]:
def check_file_exists(path):
    """Stop the notebook with a clear message if a required file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return path


def drop_spatial_ref(ds):
    """Remove the auxiliary spatial reference variable when present."""
    return ds.drop_vars("spatial_ref", errors="ignore")


def open_component_dataset(path, expected_variable):
    """Open one wind component file and check that the expected variable exists."""
    path = check_file_exists(path)
    ds = xr.open_dataset(path).rio.write_crs("EPSG:4326")
    ds = drop_spatial_ref(ds)

    if expected_variable not in ds.data_vars:
        available = list(ds.data_vars)
        raise ValueError(
            f"Variable '{expected_variable}' was not found in {path}. "
            f"Available variables are: {available}"
        )

    return ds


def calculate_wind_speed_and_direction(ds):
    """Calculate wind speed and meteorological wind direction from u and v components.

    Wind speed is computed as sqrt(u**2 + v**2).

    Wind direction follows the meteorological convention:
    0 degrees means wind coming from the north;
    90 degrees means wind coming from the east;
    values increase clockwise.
    """
    u = ds[U_COMPONENT_VAR]
    v = ds[V_COMPONENT_VAR]

    ds[WIND_SPEED_VAR] = np.sqrt(u**2 + v**2)

    ds[WIND_DIRECTION_VAR] = (
        np.rad2deg(
            np.arctan2(
                -u,
                -v,
            )
        ) + 360
    ) % 360

    ds[WIND_SPEED_VAR].attrs.update({
        "long_name": "10 m wind speed",
        "units": "m s-1",
        "description": "Wind speed computed from the integrated u and v wind components.",
    })

    ds[WIND_DIRECTION_VAR].attrs.update({
        "long_name": "10 m wind direction",
        "units": "degrees",
        "description": "Meteorological wind direction. Direction indicates where wind comes from.",
    })

    return ds


def save_xarray_netcdf_fast(ds, output_path):
    """Save a NetCDF file with light compression."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {
        var: {"zlib": True, "complevel": 1}
        for var in ds.data_vars
    }

    ds.to_netcdf(
        output_path,
        engine="netcdf4",
        encoding=encoding,
    )

## 4. Convert one month

This function reads the integrated u and v components for one month, merges them into a single dataset, computes wind speed and direction, and saves the final NetCDF file.

The output file is named like this:

```text
10m_wind_integrated_chile_m11_continental.nc
```

In [5]:
def convert_month(month):
    """Convert u and v wind components into wind speed and direction for one month."""
    print(f"Processing month {month}...")

    u_file = INPUT_DIR / f"u10_integrated_{COUNTRY}_m{month}_{AREA_NAME}.nc"
    v_file = INPUT_DIR / f"v10_integrated_{COUNTRY}_m{month}_{AREA_NAME}.nc"
    output_file = OUTPUT_DIR / f"10m_wind_integrated_{COUNTRY}_m{month}_{AREA_NAME}.nc"

    print("Reading u component:", u_file)
    ds_u = open_component_dataset(u_file, U_COMPONENT_VAR)

    print("Reading v component:", v_file)
    ds_v = open_component_dataset(v_file, V_COMPONENT_VAR)

    print("Merging u and v components...")
    ds = xr.merge([ds_u, ds_v], compat="override")
    ds = ds.rio.write_crs("EPSG:4326")
    ds = drop_spatial_ref(ds)

    print("Calculating wind speed and wind direction...")
    ds = calculate_wind_speed_and_direction(ds)
    ds = ds.rio.write_crs("EPSG:4326")
    ds = drop_spatial_ref(ds)

    print("Saving converted wind dataset:", output_file)
    save_xarray_netcdf_fast(ds, output_file)

    ds.close()
    ds_u.close()
    ds_v.close()

    return output_file

## 5. Run the conversion

Run this cell to process all months listed in `MONTHS`.

If one input file is missing, the notebook stops with a clear error message showing the missing path.

In [6]:
converted_files = []

for month in MONTHS:
    converted_file = convert_month(month)
    converted_files.append(converted_file)

print("Conversion completed.")
print("Generated files:")
for converted_file in converted_files:
    print(converted_file)

Processing month 1...
Reading u component: ../data/atlas_data/chile/sub_areas/u10_integrated_chile_m1_continental.nc


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Reading v component: ../data/atlas_data/chile/sub_areas/v10_integrated_chile_m1_continental.nc
Merging u and v components...
Calculating wind speed and wind direction...
Saving converted wind dataset: ../data/atlas_data/chile/10m_wind_integrated_chile_m1_continental.nc
Conversion completed.
Generated files:
../data/atlas_data/chile/10m_wind_integrated_chile_m1_continental.nc


## 6. Quick output check

This optional cell opens the first generated file and prints its content.

Use it to confirm that the output contains `wds_integrated` and `dir_integrated` before moving to the plotting notebook.

In [7]:
if converted_files:
    check_ds = xr.open_dataset(converted_files[0])
    print(check_ds)
    check_ds.close()

<xarray.Dataset> Size: 8GB
Dimensions:         (latitude: 46848, longitude: 10554)
Coordinates:
  * latitude        (latitude) float32 187kB -17.5 -17.5 -17.5 ... -56.54 -56.54
  * longitude       (longitude) float32 42kB -75.72 -75.72 ... -66.93 -66.93
Data variables:
    u10_integrated  (latitude, longitude) float32 2GB ...
    v10_integrated  (latitude, longitude) float32 2GB ...
    wds_integrated  (latitude, longitude) float32 2GB ...
    dir_integrated  (latitude, longitude) float32 2GB ...
